# Introduction to deep learning- Assignment 1


In [1]:
from typing import Any, Literal

from torch.nn.functional import conv2d
from tqdm.auto import tqdm,trange
import torch
from torchmetrics.classification import MulticlassAccuracy, MulticlassPrecision, MulticlassF1Score
from torch import nn
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
from torch.utils.data import Dataset
from torchvision.transforms import v2
import matplotlib.pyplot as plt
import torch.nn.functional as F
import os

device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")
print(torch.cuda.get_device_name(device)
)

Using cuda device
AMD Radeon RX 7600S


# Loading Training and Test sets from PyTorch

In [2]:
target_transform = v2.Lambda(
    lambda y: F.one_hot(torch.tensor(y), num_classes=10).float()
)
training_data = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)]),
    #target_transform=target_transform
)

test_data = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)]),
    #target_transform=target_transform
)

Trainig_data_ci = datasets.CIFAR10(
    root="data",
    train=True,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)]),
    #target_transform=target_transform

)

test_data_ci = datasets.CIFAR10(
    root="data",
    train=False,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)]),
    #target_transform=target_transform
)

# MLP class using 2 hidden layers

In [3]:
def edit_stack(stack, position, layer=None, replace=False):
    """Return a new nn.Sequential where `layer` is inserted at `position`.
    replace=True  -> the layer at `position` is replaced instead
    layer=None    -> the layer at `position` is removed"""
    layers = list(stack)
    if layer is None or replace:
        if not 0 <= position < len(layers):
            raise IndexError(f"There is no layer at position {position} (positions are 0..{len(layers) - 1})")
    elif not 0 <= position <= len(layers):
        raise IndexError(f"Can't insert at position {position} (positions are 0..{len(layers)})")

    if layer is None:
        del layers[position]
    elif replace:
        layers[position] = layer
    else:
        layers.insert(position, layer)     # inserting never overwrites anything
    return nn.Sequential(*layers)


class NeuralNetwork(nn.Module):
    def __init__(self, x=0.0, y=0.0, input=28*28, first_hidden=512, second_hidden=512,
                 first_activation=nn.ReLU, second_activation=nn.ReLU, layers=None):
        super().__init__()
        self.input = input
        self.flatten = nn.Flatten()
        if layers is not None:                       # option 2: pass nn.Sequential
            self.linear_stack = layers
        else:
            self.linear_stack = nn.Sequential(
                nn.Linear(input, first_hidden),
                first_activation(),
                nn.Dropout(x),
                nn.Linear(first_hidden, second_hidden),
                second_activation(),
                nn.Dropout(y),
                nn.Linear(second_hidden, 10),
            )
        self.check_shapes(self.linear_stack)

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_stack(x)
        return logits

    def check_shapes(self, stack):
        """Push a dummy input through the stack; fail early if the sizes don't fit."""
        device = next(stack.parameters()).device
        try:
            with torch.no_grad():
                out = stack(torch.zeros(2, self.input, device=device))
        except RuntimeError as e:
            raise ValueError(f"The layer sizes don't fit together: {e}") from None
        if out.shape[1] != 10:
            raise ValueError(f"The last layer must output 10 classes, it outputs {out.shape[1]}")

    def add_layer(self, position, layer, replace=False):
        new_stack = edit_stack(self.linear_stack, position, layer, replace)
        self.check_shapes(new_stack)                 # only applied if the check passes
        self.linear_stack = new_stack

    def remove_layer(self, position):
        new_stack = edit_stack(self.linear_stack, position)
        self.check_shapes(new_stack)
        self.linear_stack = new_stack

# Weight initialization

In [4]:
def init_weights(model, weight_init=None, bias_init=None):
    for m in model.modules():
        if isinstance(m, nn.Linear):
            if weight_init is not None:
                weight_init(m.weight)
            if bias_init is not None:
                bias_init(m.bias)

# Training with MLP using Train and Validation split (with default ratio of 80/20)

In [5]:
def train_model(**kwargs):
    """Function to do the training by getting model and training set as input also gets other parameters"""
    #unpacking the arguments
    model         = kwargs.get("model", NeuralNetwork())
    train_dataset = kwargs.get("train_dataset", training_data)
    seed          = kwargs.get("seed", 528)
    epoches       = kwargs.get("epoches", 15)
    lr            = kwargs.get("lr", 0.003)
    optimizer_cls = kwargs.get("optimizer_cls", torch.optim.Adam)
    loss_FM       = kwargs.get("loss_fn", nn.CrossEntropyLoss())
    val_ratio     = kwargs.get("val_ratio", 0.2)
    batch_size    = kwargs.get("batch_size", 64)
    num_classes   = kwargs.get("num_classes", 10)
    average       = kwargs.get("average", "macro")
    save_path     = kwargs.get("save_path", "best_model.pt")
    dev           = kwargs.get("device", device)
    weight_init   = kwargs.get("weight_init", None)
    bias_init     = kwargs.get("bias_init", None)
    optimizer_kwargs = kwargs.get("optimizer_kwargs", {})
    os.makedirs("weights_saved", exist_ok=True)
    save_path = "weights_saved/"+ save_path
    # To record how the training goes and then use this dictionary to draw plots
    history = {
        "train_loss": [], "train_acc": [], "train_prec": [], "train_f1": [],
        "val_loss": [],   "val_acc": [],   "val_prec": [],   "val_f1": [],
    }

    best_val_acc = 0.0
    # Metrics used for observation and evaluation of training
    acc  = MulticlassAccuracy(num_classes=num_classes).to(dev)
    prec = MulticlassPrecision(num_classes=num_classes, average=average).to(dev)
    f1   = MulticlassF1Score(num_classes=num_classes, average=average).to(dev)
    # weight initialization
    init_weights(model, weight_init, bias_init)
    #defining the optimizer and passing the values
    optimizer = optimizer_cls(model.parameters(), lr=lr, **optimizer_kwargs)
    #making the training and validation split with percentage of the whole training dataset
    train_split, val_split = random_split(train_dataset, [len(train_dataset) - int(len(train_dataset) * val_ratio), int(len(train_dataset) * val_ratio)], generator=torch.Generator().manual_seed(seed))
    train_loader = DataLoader(dataset=train_split, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(dataset=val_split, batch_size=batch_size, shuffle=True)

    train_bar = tqdm(total=len(train_loader), desc="train")
    val_bar   = tqdm(total=len(val_loader),   desc="val")
    # Training loop
    for epoch in trange(epoches, desc="Training"):

        train_loss_total = 0.0
        val_loss_total = 0.0
        model.train().to(dev)

        train_bar.reset()
        train_bar.set_description(f"epoch {epoch} train")
        for x, y in train_loader:
            x = x.to(dev)
            y = y.to(dev)
            logits = model(x)
            loss = loss_FM(logits, y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            preds = logits.argmax(dim=1)
            acc_v = acc(preds, y)
            prec_v = prec(preds, y)
            f1_v = f1(preds, y)
            train_loss_total += loss.item()
            train_bar.set_postfix(loss=f"{loss.item():.4f}", acc=f"{acc_v:.4f}",
                                  prec=f"{prec_v:.4f}", f1=f"{f1_v:.4f}")
            train_bar.update(1)

        history["train_loss"].append(train_loss_total / len(train_loader))
        history["train_acc"].append(acc.compute().item())
        history["train_prec"].append(prec.compute().item())
        history["train_f1"].append(f1.compute().item())

        tqdm.write(f"epoch {epoch} train | loss {history['train_loss'][-1]:.4f} | "
                   f"Accuracy {history['train_acc'][-1]:.4f} | Precision {history['train_prec'][-1]:.4f} | "
                   f"F1 {history['train_f1'][-1]:.4f}")
        acc.reset()
        prec.reset()
        f1.reset()
        # each epoch we evaluate using validation set and save the highest accuracy model on validation set
        with torch.no_grad():
            model.eval()
            val_bar.reset()
            val_bar.set_description(f"epoch {epoch} val")
            for x, y in val_loader:
                x = x.to(dev)
                y = y.to(dev)
                logits = model(x)
                loss = loss_FM(logits, y)
                preds = logits.argmax(dim=1)
                acc_v = acc(preds, y)
                prec_v = prec(preds, y)
                f1_v = f1(preds, y)
                val_loss_total += loss.item()
                val_bar.set_postfix(loss=f"{loss.item():.4f}", acc=f"{acc_v:.4f}")
                val_bar.update(1)

        history["val_loss"].append(val_loss_total / len(val_loader))
        history["val_acc"].append(acc.compute().item())
        history["val_prec"].append(prec.compute().item())
        history["val_f1"].append(f1.compute().item())

        tqdm.write(f"epoch {epoch} val   | loss {history['val_loss'][-1]:.4f} | "
                   f"Accuracy {history['val_acc'][-1]:.4f} | Precision {history['val_prec'][-1]:.4f} | "
                   f"F1 {history['val_f1'][-1]:.4f}")
        if history["val_acc"][-1] > best_val_acc:
            best_val_acc = history["val_acc"][-1]
            torch.save(model.state_dict(), save_path)
            tqdm.write(f"new best val acc {best_val_acc:.4f}, saved")

        acc.reset()
        prec.reset()
        f1.reset()

    train_bar.close()
    val_bar.close()

    return history, best_val_acc

# Evaluation function to be used with test set

In [6]:
def evaluate_model(**kwargs):
    """Function to evaluate the model with test set"""
    test_model   = kwargs.get("model", NeuralNetwork())
    test_dataset = kwargs.get("test_dataset", test_data)
    load_path    = kwargs.get("load_path", "best_model.pt")
    batch_size   = kwargs.get("batch_size", 64)
    loss_FM      = kwargs.get("loss_fn", nn.CrossEntropyLoss())
    num_classes  = kwargs.get("num_classes", 10)
    average      = kwargs.get("average", "macro")
    dev          = kwargs.get("device", device)

    acc  = MulticlassAccuracy(num_classes=num_classes).to(dev)
    prec = MulticlassPrecision(num_classes=num_classes, average=average).to(dev)
    f1   = MulticlassF1Score(num_classes=num_classes, average=average).to(dev)

    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    test_model.load_state_dict(torch.load(load_path, map_location=dev))
    test_model.to(dev)
    test_model.eval()

    acc.reset(); prec.reset(); f1.reset()
    test_loss_total = 0.0

    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(dev), y.to(dev)
            logits = test_model(x)
            test_loss_total += loss_FM(logits, y).item()
            preds = logits.argmax(dim=1)
            acc.update(preds, y)
            prec.update(preds, y)
            f1.update(preds, y)

    print(f"test | loss {test_loss_total/len(test_loader):.4f} | "
          f"Accuracy {acc.compute():.4f} | Precision {prec.compute():.4f} | F1 {f1.compute():.4f}")

    return {
        "loss": test_loss_total / len(test_loader),
        "acc": acc.compute().item(),
        "prec": prec.compute().item(),
        "f1": f1.compute().item(),
    }

# Make a plot of training, validation and test set metrics

In [7]:
def plot_history(history, test=None, title=""):
    """Plot train and validation curves per epoch (from train_model) for loss, accuracy,
    precision and F1. The test result (from evaluate_model) is drawn as a dashed line."""
    metrics = [("loss", "Loss"), ("acc", "Accuracy"), ("prec", "Precision"), ("f1", "F1")]
    epochs = range(1, len(history["train_loss"]) + 1)

    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    for ax, (key, name) in zip(axes.flat, metrics):
        ax.plot(epochs, history[f"train_{key}"], marker="o", label="train")
        ax.plot(epochs, history[f"val_{key}"], marker="o", label="validation")
        if test is not None:
            ax.axhline(test[key], color="red", linestyle="--", label=f"test ({test[key]:.3f})")
        ax.set_title(name)
        ax.set_xlabel("epoch")
        ax.grid(alpha=0.3)
        ax.legend()
    fig.suptitle(title)
    fig.tight_layout()
    plt.show()

# Run 1 without dropout (FashionMNIST)

In [7]:
# Run 1: no dropout
model_mlp = NeuralNetwork()
history, best = train_model(model=model_mlp, save_path="best_model.pt")
evaluate_model(model=model_mlp, load_path="weights_saved/best_model.pt")

train:   0%|          | 0/750 [00:00<?, ?it/s]

val:   0%|          | 0/188 [00:00<?, ?it/s]

Training:   0%|          | 0/15 [00:00<?, ?it/s]

epoch 0 train | loss 0.5135 | Accuracy 0.8123 | Precision 0.8105 | F1 0.8103
epoch 0 val   | loss 0.4184 | Accuracy 0.8489 | Precision 0.8538 | F1 0.8498
new best val acc 0.8489, saved
epoch 1 train | loss 0.3836 | Accuracy 0.8610 | Precision 0.8606 | F1 0.8603
epoch 1 val   | loss 0.3918 | Accuracy 0.8579 | Precision 0.8620 | F1 0.8571
new best val acc 0.8579, saved
epoch 2 train | loss 0.3452 | Accuracy 0.8732 | Precision 0.8728 | F1 0.8726
epoch 2 val   | loss 0.3712 | Accuracy 0.8591 | Precision 0.8664 | F1 0.8598
new best val acc 0.8591, saved
epoch 3 train | loss 0.3181 | Accuracy 0.8831 | Precision 0.8828 | F1 0.8827
epoch 3 val   | loss 0.3556 | Accuracy 0.8729 | Precision 0.8744 | F1 0.8730
new best val acc 0.8729, saved
epoch 4 train | loss 0.3034 | Accuracy 0.8872 | Precision 0.8869 | F1 0.8868
epoch 4 val   | loss 0.3719 | Accuracy 0.8715 | Precision 0.8752 | F1 0.8708
epoch 5 train | loss 0.2910 | Accuracy 0.8921 | Precision 0.8918 | F1 0.8918
epoch 5 val   | loss 0.3498 |

{'loss': 0.38265409605328443,
 'acc': 0.8806999921798706,
 'prec': 0.879787802696228,
 'f1': 0.8795100450515747}

# Run 2 with dropout 0.2 (FashionMNIST)

In [8]:
# Run 2: dropout 0.2 / 0.2
model_mlp_drop = NeuralNetwork(x=0.2, y=0.2)
history_drop, best_drop = train_model(model=model_mlp_drop, save_path="best_model_drop.pt")
evaluate_model(model=model_mlp_drop, load_path="weights_saved/best_model_drop.pt")

train:   0%|          | 0/750 [00:00<?, ?it/s]

val:   0%|          | 0/188 [00:00<?, ?it/s]

Training:   0%|          | 0/15 [00:00<?, ?it/s]

epoch 0 train | loss 0.5566 | Accuracy 0.7957 | Precision 0.7943 | F1 0.7937
epoch 0 val   | loss 0.4280 | Accuracy 0.8436 | Precision 0.8444 | F1 0.8414
new best val acc 0.8436, saved
epoch 1 train | loss 0.4403 | Accuracy 0.8388 | Precision 0.8376 | F1 0.8375
epoch 1 val   | loss 0.3902 | Accuracy 0.8592 | Precision 0.8619 | F1 0.8578
new best val acc 0.8592, saved
epoch 2 train | loss 0.4107 | Accuracy 0.8525 | Precision 0.8519 | F1 0.8518
epoch 2 val   | loss 0.3963 | Accuracy 0.8509 | Precision 0.8574 | F1 0.8506
epoch 3 train | loss 0.3974 | Accuracy 0.8570 | Precision 0.8563 | F1 0.8562
epoch 3 val   | loss 0.3776 | Accuracy 0.8668 | Precision 0.8690 | F1 0.8665
new best val acc 0.8668, saved
epoch 4 train | loss 0.3776 | Accuracy 0.8626 | Precision 0.8618 | F1 0.8618
epoch 4 val   | loss 0.3908 | Accuracy 0.8581 | Precision 0.8610 | F1 0.8583
epoch 5 train | loss 0.3751 | Accuracy 0.8629 | Precision 0.8621 | F1 0.8621
epoch 5 val   | loss 0.3772 | Accuracy 0.8614 | Precision 0.

{'loss': 0.35754424115275124,
 'acc': 0.8773000240325928,
 'prec': 0.8768547773361206,
 'f1': 0.8759050965309143}

# Function to run experiments for different parameters

In [8]:
results = {}

def run_experiment(experiment, runs, model_cls=NeuralNetwork, init_seed=528):
    for run in runs:
        name = f"{experiment}_{run['name']}"
        model_kwargs = run.get("model_kwargs", {})
        train_kwargs = run.get("train_kwargs", {})

        print(f"\n===== {name} =====")
        torch.manual_seed(init_seed)
        model = model_cls(**model_kwargs)
        history, best = train_model(model=model, save_path=f"{name}.pt", **train_kwargs)
        test = evaluate_model(model=model, load_path=f"weights_saved/{name}.pt")

        results[name] = {"history": history, "best_val_acc": best, "test": test}

# Parameter initialization (torch.nn.init)


In [11]:
run_experiment("init", [
    {"name": "default"},
    {"name": "kaiming_uniform", "train_kwargs": {
        "weight_init": lambda w:torch.nn.init.kaiming_uniform_(w, mode='fan_in', nonlinearity='relu'),
        "bias_init":   lambda b: torch.nn.init.constant_(b, 0.1)}},
    {"name": "uni", "train_kwargs": {"weight_init": lambda w:torch.nn.init.uniform_(w, a=0.0, b=1.0),
                                     "bias_init": lambda b: torch.nn.init.constant_(b, 0)}},
    {"name": "uni_biasnonzero", "train_kwargs": {"weight_init": lambda w:torch.nn.init.uniform_(w, a=0.0, b=1.0),
                                     "bias_init": lambda b: torch.nn.init.constant_(b, 0.1)}},
])


===== init_default =====


train:   0%|          | 0/750 [00:00<?, ?it/s]

val:   0%|          | 0/188 [00:00<?, ?it/s]

Training:   0%|          | 0/15 [00:00<?, ?it/s]

epoch 0 train | loss 0.5063 | Accuracy 0.8146 | Precision 0.8136 | F1 0.8131
epoch 0 val   | loss 0.4163 | Accuracy 0.8469 | Precision 0.8593 | F1 0.8501
new best val acc 0.8469, saved
epoch 1 train | loss 0.3828 | Accuracy 0.8627 | Precision 0.8624 | F1 0.8622
epoch 1 val   | loss 0.3766 | Accuracy 0.8651 | Precision 0.8684 | F1 0.8625
new best val acc 0.8651, saved
epoch 2 train | loss 0.3430 | Accuracy 0.8738 | Precision 0.8733 | F1 0.8733
epoch 2 val   | loss 0.3653 | Accuracy 0.8702 | Precision 0.8728 | F1 0.8680
new best val acc 0.8702, saved
epoch 3 train | loss 0.3216 | Accuracy 0.8813 | Precision 0.8809 | F1 0.8809
epoch 3 val   | loss 0.3583 | Accuracy 0.8714 | Precision 0.8743 | F1 0.8713
new best val acc 0.8714, saved
epoch 4 train | loss 0.3079 | Accuracy 0.8852 | Precision 0.8848 | F1 0.8848
epoch 4 val   | loss 0.3510 | Accuracy 0.8717 | Precision 0.8738 | F1 0.8718
new best val acc 0.8717, saved
epoch 5 train | loss 0.2925 | Accuracy 0.8907 | Precision 0.8904 | F1 0.890

...

epoch 0 train | loss 7706.8702 | Accuracy 0.4243 | Precision 0.4283 | F1 0.4240
epoch 0 val   | loss 2524.3623 | Accuracy 0.5181 | Precision 0.5503 | F1 0.4618
new best val acc 0.5181, saved
epoch 1 train | loss 794.8559 | Accuracy 0.6723 | Precision 0.6707 | F1 0.6709
epoch 1 val   | loss 413.3469 | Accuracy 0.7257 | Precision 0.7683 | F1 0.7102
new best val acc 0.7257, saved
epoch 2 train | loss 430.6646 | Accuracy 0.7462 | Precision 0.7452 | F1 0.7455
epoch 2 val   | loss 166.9701 | Accuracy 0.8130 | Precision 0.8258 | F1 0.8163
new best val acc 0.8130, saved
epoch 3 train | loss 218.4186 | Accuracy 0.7868 | Precision 0.7866 | F1 0.7866
epoch 3 val   | loss 148.4275 | Accuracy 0.7828 | Precision 0.8141 | F1 0.7778
epoch 4 train | loss 98.9000 | Accuracy 0.8058 | Precision 0.8058 | F1 0.8057
epoch 4 val   | loss 65.0532 | Accuracy 0.8114 | Precision 0.8429 | F1 0.8107
epoch 5 train | loss 43.8470 | Accuracy 0.8184 | Precision 0.8186 | F1 0.8185
epoch 5 val   | loss 30.3094 | Accuracy

train:   0%|          | 0/750 [00:00<?, ?it/s]

val:   0%|          | 0/188 [00:00<?, ?it/s]

Training:   0%|          | 0/15 [00:00<?, ?it/s]

epoch 0 train | loss 7555.1737 | Accuracy 0.4200 | Precision 0.4231 | F1 0.4192
epoch 0 val   | loss 2081.6004 | Accuracy 0.5743 | Precision 0.6700 | F1 0.5246
new best val acc 0.5743, saved
epoch 1 train | loss 858.4639 | Accuracy 0.6691 | Precision 0.6674 | F1 0.6676
epoch 1 val   | loss 446.1501 | Accuracy 0.7378 | Precision 0.8203 | F1 0.7390
new best val acc 0.7378, saved
epoch 2 train | loss 416.6943 | Accuracy 0.7512 | Precision 0.7502 | F1 0.7505
epoch 2 val   | loss 189.5769 | Accuracy 0.8245 | Precision 0.8329 | F1 0.8226
new best val acc 0.8245, saved
epoch 3 train | loss 218.2636 | Accuracy 0.7879 | Precision 0.7877 | F1 0.7877
epoch 3 val   | loss 162.0470 | Accuracy 0.7717 | Precision 0.8119 | F1 0.7678
epoch 4 train | loss 104.4213 | Accuracy 0.8017 | Precision 0.8017 | F1 0.8017
epoch 4 val   | loss 62.2519 | Accuracy 0.8210 | Precision 0.8393 | F1 0.8210
epoch 5 train | loss 44.3059 | Accuracy 0.8193 | Precision 0.8195 | F1 0.8194
epoch 5 val   | loss 36.3599 | Accurac

# Activation functions

In [12]:

run_experiment("activation", [
    {"name": "relu"},
    {"name": "LeakyReLU", "model_kwargs": {"first_activation": nn.LeakyReLU, "second_activation": nn.LeakyReLU}},
    {"name": "nn.GELU", "model_kwargs": {"first_activation": nn.GELU, "second_activation": nn.GELU}},
    {"name": "combination1", "model_kwargs": {"first_activation": nn.ReLU, "second_activation": nn.GELU}},
    {"name": "combination2", "model_kwargs": {"first_activation": nn.GELU, "second_activation": nn.LeakyReLU}},
])


===== activation_relu =====


train:   0%|          | 0/750 [00:00<?, ?it/s]

val:   0%|          | 0/188 [00:00<?, ?it/s]

Training:   0%|          | 0/15 [00:00<?, ?it/s]

epoch 0 train | loss 0.5063 | Accuracy 0.8146 | Precision 0.8136 | F1 0.8131
epoch 0 val   | loss 0.4163 | Accuracy 0.8469 | Precision 0.8593 | F1 0.8501
new best val acc 0.8469, saved
epoch 1 train | loss 0.3828 | Accuracy 0.8627 | Precision 0.8624 | F1 0.8622
epoch 1 val   | loss 0.3766 | Accuracy 0.8651 | Precision 0.8684 | F1 0.8625
new best val acc 0.8651, saved
epoch 2 train | loss 0.3430 | Accuracy 0.8738 | Precision 0.8733 | F1 0.8733
epoch 2 val   | loss 0.3653 | Accuracy 0.8702 | Precision 0.8728 | F1 0.8680
new best val acc 0.8702, saved
epoch 3 train | loss 0.3216 | Accuracy 0.8813 | Precision 0.8809 | F1 0.8809
epoch 3 val   | loss 0.3583 | Accuracy 0.8714 | Precision 0.8743 | F1 0.8713
new best val acc 0.8714, saved
epoch 4 train | loss 0.3079 | Accuracy 0.8852 | Precision 0.8848 | F1 0.8848
epoch 4 val   | loss 0.3510 | Accuracy 0.8717 | Precision 0.8738 | F1 0.8718
new best val acc 0.8717, saved
epoch 5 train | loss 0.2925 | Accuracy 0.8907 | Precision 0.8904 | F1 0.890

...

epoch 0 train | loss 0.5031 | Accuracy 0.8146 | Precision 0.8128 | F1 0.8130
epoch 0 val   | loss 0.4284 | Accuracy 0.8398 | Precision 0.8608 | F1 0.8428
new best val acc 0.8398, saved
epoch 1 train | loss 0.3801 | Accuracy 0.8610 | Precision 0.8604 | F1 0.8604
epoch 1 val   | loss 0.3747 | Accuracy 0.8677 | Precision 0.8695 | F1 0.8642
new best val acc 0.8677, saved
epoch 2 train | loss 0.3445 | Accuracy 0.8738 | Precision 0.8735 | F1 0.8734
epoch 2 val   | loss 0.3534 | Accuracy 0.8703 | Precision 0.8711 | F1 0.8691
new best val acc 0.8703, saved
epoch 3 train | loss 0.3213 | Accuracy 0.8809 | Precision 0.8804 | F1 0.8804
epoch 3 val   | loss 0.3438 | Accuracy 0.8722 | Precision 0.8739 | F1 0.8719
new best val acc 0.8722, saved
epoch 4 train | loss 0.3054 | Accuracy 0.8869 | Precision 0.8865 | F1 0.8866
epoch 4 val   | loss 0.3387 | Accuracy 0.8747 | Precision 0.8758 | F1 0.8734
new best val acc 0.8747, saved
epoch 5 train | loss 0.2913 | Accuracy 0.8912 | Precision 0.8907 | F1 0.890

train:   0%|          | 0/750 [00:00<?, ?it/s]

val:   0%|          | 0/188 [00:00<?, ?it/s]

Training:   0%|          | 0/15 [00:00<?, ?it/s]

epoch 0 train | loss 0.4931 | Accuracy 0.8172 | Precision 0.8154 | F1 0.8152
epoch 0 val   | loss 0.4324 | Accuracy 0.8359 | Precision 0.8579 | F1 0.8383
new best val acc 0.8359, saved
epoch 1 train | loss 0.3736 | Accuracy 0.8648 | Precision 0.8643 | F1 0.8641
epoch 1 val   | loss 0.3670 | Accuracy 0.8631 | Precision 0.8668 | F1 0.8609
new best val acc 0.8631, saved
epoch 2 train | loss 0.3415 | Accuracy 0.8750 | Precision 0.8746 | F1 0.8745
epoch 2 val   | loss 0.3584 | Accuracy 0.8718 | Precision 0.8715 | F1 0.8700
new best val acc 0.8718, saved
epoch 3 train | loss 0.3126 | Accuracy 0.8828 | Precision 0.8826 | F1 0.8824
epoch 3 val   | loss 0.4016 | Accuracy 0.8593 | Precision 0.8630 | F1 0.8572
epoch 4 train | loss 0.3012 | Accuracy 0.8880 | Precision 0.8875 | F1 0.8874
epoch 4 val   | loss 0.3571 | Accuracy 0.8672 | Precision 0.8687 | F1 0.8663
epoch 5 train | loss 0.2872 | Accuracy 0.8942 | Precision 0.8939 | F1 0.8938
epoch 5 val   | loss 0.3747 | Accuracy 0.8620 | Precision 0.

# Optimizers

In [13]:
run_experiment("optimizer", [
    {"name": "SGD", "train_kwargs": {"optimizer_cls": torch.optim.SGD, "lr": 0.005,
                                     "optimizer_kwargs": {"momentum": 0.89}}},
    {"name": "AdamW", "train_kwargs": {"optimizer_cls": torch.optim.AdamW, "lr": 0.005,
                                       "optimizer_kwargs": {"weight_decay": 0.05}}},
    {"name": "Adam", "train_kwargs": {"optimizer_cls": torch.optim.Adam, "lr": 0.005}},
])


===== optimizer_SGD =====


train:   0%|          | 0/750 [00:00<?, ?it/s]

val:   0%|          | 0/188 [00:00<?, ?it/s]

Training:   0%|          | 0/15 [00:00<?, ?it/s]

epoch 0 train | loss 0.8566 | Accuracy 0.7034 | Precision 0.7149 | F1 0.6989
epoch 0 val   | loss 0.5424 | Accuracy 0.8113 | Precision 0.8237 | F1 0.8135
new best val acc 0.8113, saved
epoch 1 train | loss 0.4933 | Accuracy 0.8266 | Precision 0.8244 | F1 0.8249
epoch 1 val   | loss 0.4556 | Accuracy 0.8457 | Precision 0.8456 | F1 0.8428
new best val acc 0.8457, saved
epoch 2 train | loss 0.4374 | Accuracy 0.8443 | Precision 0.8429 | F1 0.8432
epoch 2 val   | loss 0.4325 | Accuracy 0.8504 | Precision 0.8531 | F1 0.8464
new best val acc 0.8504, saved
epoch 3 train | loss 0.4020 | Accuracy 0.8576 | Precision 0.8564 | F1 0.8567
epoch 3 val   | loss 0.3957 | Accuracy 0.8609 | Precision 0.8607 | F1 0.8603
new best val acc 0.8609, saved
epoch 4 train | loss 0.3804 | Accuracy 0.8647 | Precision 0.8636 | F1 0.8639
epoch 4 val   | loss 0.3906 | Accuracy 0.8626 | Precision 0.8645 | F1 0.8615
new best val acc 0.8626, saved
epoch 5 train | loss 0.3600 | Accuracy 0.8716 | Precision 0.8707 | F1 0.870

...

epoch 0 train | loss 0.5236 | Accuracy 0.8072 | Precision 0.8056 | F1 0.8052
epoch 0 val   | loss 0.4493 | Accuracy 0.8425 | Precision 0.8473 | F1 0.8423
new best val acc 0.8425, saved
epoch 1 train | loss 0.4083 | Accuracy 0.8525 | Precision 0.8517 | F1 0.8516
epoch 1 val   | loss 0.3882 | Accuracy 0.8587 | Precision 0.8610 | F1 0.8562
new best val acc 0.8587, saved
epoch 2 train | loss 0.3755 | Accuracy 0.8628 | Precision 0.8621 | F1 0.8621
epoch 2 val   | loss 0.3925 | Accuracy 0.8587 | Precision 0.8637 | F1 0.8571
new best val acc 0.8587, saved
epoch 3 train | loss 0.3642 | Accuracy 0.8660 | Precision 0.8655 | F1 0.8654
epoch 3 val   | loss 0.3733 | Accuracy 0.8661 | Precision 0.8646 | F1 0.8634
new best val acc 0.8661, saved
epoch 4 train | loss 0.3504 | Accuracy 0.8724 | Precision 0.8717 | F1 0.8717
epoch 4 val   | loss 0.3853 | Accuracy 0.8587 | Precision 0.8612 | F1 0.8585
epoch 5 train | loss 0.3460 | Accuracy 0.8739 | Precision 0.8733 | F1 0.8732
epoch 5 val   | loss 0.3974 |

train:   0%|          | 0/750 [00:00<?, ?it/s]

val:   0%|          | 0/188 [00:00<?, ?it/s]

Training:   0%|          | 0/15 [00:00<?, ?it/s]

epoch 0 train | loss 0.5215 | Accuracy 0.8102 | Precision 0.8085 | F1 0.8083
epoch 0 val   | loss 0.4134 | Accuracy 0.8485 | Precision 0.8582 | F1 0.8511
new best val acc 0.8485, saved
epoch 1 train | loss 0.3976 | Accuracy 0.8572 | Precision 0.8564 | F1 0.8564
epoch 1 val   | loss 0.4125 | Accuracy 0.8545 | Precision 0.8585 | F1 0.8509
new best val acc 0.8545, saved
epoch 2 train | loss 0.3603 | Accuracy 0.8692 | Precision 0.8687 | F1 0.8687
epoch 2 val   | loss 0.3628 | Accuracy 0.8703 | Precision 0.8713 | F1 0.8702
new best val acc 0.8703, saved
epoch 3 train | loss 0.3405 | Accuracy 0.8755 | Precision 0.8749 | F1 0.8750
epoch 3 val   | loss 0.3592 | Accuracy 0.8706 | Precision 0.8695 | F1 0.8690
new best val acc 0.8706, saved
epoch 4 train | loss 0.3296 | Accuracy 0.8791 | Precision 0.8786 | F1 0.8786
epoch 4 val   | loss 0.3665 | Accuracy 0.8644 | Precision 0.8666 | F1 0.8645
epoch 5 train | loss 0.3191 | Accuracy 0.8827 | Precision 0.8822 | F1 0.8822
epoch 5 val   | loss 0.4117 |

# Architecture: width

In [9]:
run_experiment("width", [
    {"name": "128", "model_kwargs": {"first_hidden": 128, "second_hidden": 128}},
    {"name": "256", "model_kwargs": {"first_hidden": 256, "second_hidden": 256}},
    {"name": "784", "model_kwargs": {"first_hidden": 784, "second_hidden": 784}},
])


===== width_128 =====


train:   0%|          | 0/750 [00:00<?, ?it/s]

val:   0%|          | 0/188 [00:00<?, ?it/s]

Training:   0%|          | 0/15 [00:00<?, ?it/s]

epoch 0 train | loss 0.5260 | Accuracy 0.8062 | Precision 0.8049 | F1 0.8044
epoch 0 val   | loss 0.4043 | Accuracy 0.8541 | Precision 0.8568 | F1 0.8511
new best val acc 0.8541, saved
epoch 1 train | loss 0.3830 | Accuracy 0.8595 | Precision 0.8588 | F1 0.8588
epoch 1 val   | loss 0.4194 | Accuracy 0.8471 | Precision 0.8669 | F1 0.8426
epoch 2 train | loss 0.3449 | Accuracy 0.8740 | Precision 0.8735 | F1 0.8735
epoch 2 val   | loss 0.3628 | Accuracy 0.8716 | Precision 0.8747 | F1 0.8689
new best val acc 0.8716, saved
epoch 3 train | loss 0.3240 | Accuracy 0.8806 | Precision 0.8801 | F1 0.8801
epoch 3 val   | loss 0.3674 | Accuracy 0.8635 | Precision 0.8682 | F1 0.8632
epoch 4 train | loss 0.3080 | Accuracy 0.8846 | Precision 0.8841 | F1 0.8842
epoch 4 val   | loss 0.3421 | Accuracy 0.8724 | Precision 0.8738 | F1 0.8708
new best val acc 0.8724, saved
epoch 5 train | loss 0.2940 | Accuracy 0.8899 | Precision 0.8895 | F1 0.8895
epoch 5 val   | loss 0.3597 | Accuracy 0.8729 | Precision 0.

...

epoch 0 train | loss 0.5152 | Accuracy 0.8122 | Precision 0.8106 | F1 0.8105
epoch 0 val   | loss 0.4199 | Accuracy 0.8424 | Precision 0.8475 | F1 0.8425
new best val acc 0.8424, saved
epoch 1 train | loss 0.3847 | Accuracy 0.8592 | Precision 0.8586 | F1 0.8585
epoch 1 val   | loss 0.3807 | Accuracy 0.8646 | Precision 0.8661 | F1 0.8624
new best val acc 0.8646, saved
epoch 2 train | loss 0.3473 | Accuracy 0.8730 | Precision 0.8725 | F1 0.8725
epoch 2 val   | loss 0.3491 | Accuracy 0.8702 | Precision 0.8697 | F1 0.8684
new best val acc 0.8702, saved
epoch 3 train | loss 0.3199 | Accuracy 0.8815 | Precision 0.8813 | F1 0.8811
epoch 3 val   | loss 0.3514 | Accuracy 0.8751 | Precision 0.8754 | F1 0.8742
new best val acc 0.8751, saved
epoch 4 train | loss 0.3039 | Accuracy 0.8880 | Precision 0.8877 | F1 0.8877
epoch 4 val   | loss 0.3489 | Accuracy 0.8740 | Precision 0.8772 | F1 0.8734
epoch 5 train | loss 0.2908 | Accuracy 0.8908 | Precision 0.8904 | F1 0.8904
epoch 5 val   | loss 0.3527 |

train:   0%|          | 0/750 [00:00<?, ?it/s]

val:   0%|          | 0/188 [00:00<?, ?it/s]

Training:   0%|          | 0/15 [00:00<?, ?it/s]

epoch 0 train | loss 0.5107 | Accuracy 0.8144 | Precision 0.8130 | F1 0.8127
epoch 0 val   | loss 0.4126 | Accuracy 0.8473 | Precision 0.8495 | F1 0.8447
new best val acc 0.8473, saved
epoch 1 train | loss 0.3839 | Accuracy 0.8592 | Precision 0.8584 | F1 0.8585
epoch 1 val   | loss 0.3898 | Accuracy 0.8578 | Precision 0.8589 | F1 0.8559
new best val acc 0.8578, saved
epoch 2 train | loss 0.3412 | Accuracy 0.8746 | Precision 0.8741 | F1 0.8741
epoch 2 val   | loss 0.3492 | Accuracy 0.8742 | Precision 0.8742 | F1 0.8725
new best val acc 0.8742, saved
epoch 3 train | loss 0.3228 | Accuracy 0.8801 | Precision 0.8796 | F1 0.8796
epoch 3 val   | loss 0.3947 | Accuracy 0.8601 | Precision 0.8670 | F1 0.8605
epoch 4 train | loss 0.2999 | Accuracy 0.8889 | Precision 0.8887 | F1 0.8886
epoch 4 val   | loss 0.3590 | Accuracy 0.8712 | Precision 0.8712 | F1 0.8679
epoch 5 train | loss 0.2882 | Accuracy 0.8920 | Precision 0.8917 | F1 0.8916
epoch 5 val   | loss 0.3413 | Accuracy 0.8797 | Precision 0.

In [10]:
for name, r in results.items():
    t = r["test"]
    print(f"{name:30s} | best val acc {r['best_val_acc']:.4f} | test loss {t['loss']:.4f} | "
          f"test acc {t['acc']:.4f} | prec {t['prec']:.4f} | f1 {t['f1']:.4f}")

width_128                      | best val acc 0.8866 | test loss 0.3539 | test acc 0.8811 | prec 0.8820 | f1 0.8812
width_256                      | best val acc 0.8864 | test loss 0.3660 | test acc 0.8802 | prec 0.8795 | f1 0.8794
width_784                      | best val acc 0.8839 | test loss 0.3871 | test acc 0.8775 | prec 0.8776 | f1 0.8773


# CIFAR-10 (MLP)


In [11]:
model_mlp_ci = NeuralNetwork(input=3*32*32, first_hidden=3200, second_hidden=3000)
history_ci, best_ci = train_model(model=model_mlp_ci, save_path="best_model_Cifar.pt", train_dataset=Trainig_data_ci, epoches=25)
evaluate_model(model=model_mlp_ci, load_path="weights_saved/best_model_Cifar.pt", test_dataset=test_data_ci)

train:   0%|          | 0/625 [00:00<?, ?it/s]

val:   0%|          | 0/157 [00:00<?, ?it/s]

Training:   0%|          | 0/25 [00:00<?, ?it/s]

epoch 0 train | loss 2.1264 | Accuracy 0.2846 | Precision 0.2760 | F1 0.2765
epoch 0 val   | loss 1.8551 | Accuracy 0.3322 | Precision 0.3513 | F1 0.3157
new best val acc 0.3322, saved
epoch 1 train | loss 1.7939 | Accuracy 0.3470 | Precision 0.3409 | F1 0.3409
epoch 1 val   | loss 1.7536 | Accuracy 0.3660 | Precision 0.3629 | F1 0.3430
new best val acc 0.3660, saved
epoch 2 train | loss 1.7381 | Accuracy 0.3723 | Precision 0.3659 | F1 0.3661
epoch 2 val   | loss 1.7165 | Accuracy 0.3818 | Precision 0.3806 | F1 0.3732
new best val acc 0.3818, saved
epoch 3 train | loss 1.7078 | Accuracy 0.3813 | Precision 0.3755 | F1 0.3756
epoch 3 val   | loss 1.7539 | Accuracy 0.3692 | Precision 0.3803 | F1 0.3501
epoch 4 train | loss 1.6751 | Accuracy 0.3922 | Precision 0.3860 | F1 0.3863
epoch 4 val   | loss 1.7165 | Accuracy 0.3819 | Precision 0.4037 | F1 0.3720
new best val acc 0.3819, saved
epoch 5 train | loss 1.6474 | Accuracy 0.4050 | Precision 0.3989 | F1 0.3993
epoch 5 val   | loss 1.6685 |

{'loss': 1.6030100249940422,
 'acc': 0.42570000886917114,
 'prec': 0.42695385217666626,
 'f1': 0.41552218794822693}

In [12]:
model_mlp_ci_lr = NeuralNetwork(input=3*32*32, first_hidden=3200, second_hidden=3000)
history_ci, best_ci = train_model(model=model_mlp_ci_lr, save_path="best_model_Cifar(3e-4).pt", train_dataset=Trainig_data_ci, epoches=25, lr=3e-4)
evaluate_model(model=model_mlp_ci_lr, load_path="weights_saved/best_model_Cifar(3e-4).pt", test_dataset=test_data_ci)

train:   0%|          | 0/625 [00:00<?, ?it/s]

val:   0%|          | 0/157 [00:00<?, ?it/s]

Training:   0%|          | 0/25 [00:00<?, ?it/s]

epoch 0 train | loss 1.8471 | Accuracy 0.3232 | Precision 0.3163 | F1 0.3170
epoch 0 val   | loss 1.7596 | Accuracy 0.3688 | Precision 0.3731 | F1 0.3502
new best val acc 0.3688, saved
epoch 1 train | loss 1.6629 | Accuracy 0.4024 | Precision 0.3963 | F1 0.3968
epoch 1 val   | loss 1.6229 | Accuracy 0.4217 | Precision 0.4430 | F1 0.4123
new best val acc 0.4217, saved
epoch 2 train | loss 1.5670 | Accuracy 0.4404 | Precision 0.4333 | F1 0.4345
epoch 2 val   | loss 1.6422 | Accuracy 0.4289 | Precision 0.4577 | F1 0.4278
new best val acc 0.4289, saved
epoch 3 train | loss 1.5030 | Accuracy 0.4619 | Precision 0.4552 | F1 0.4563
epoch 3 val   | loss 1.5209 | Accuracy 0.4596 | Precision 0.4619 | F1 0.4487
new best val acc 0.4596, saved
epoch 4 train | loss 1.4445 | Accuracy 0.4833 | Precision 0.4773 | F1 0.4782
epoch 4 val   | loss 1.4967 | Accuracy 0.4755 | Precision 0.4734 | F1 0.4641
new best val acc 0.4755, saved
epoch 5 train | loss 1.3958 | Accuracy 0.5009 | Precision 0.4950 | F1 0.496

{'loss': 1.4340881405362658,
 'acc': 0.5407000184059143,
 'prec': 0.5477185249328613,
 'f1': 0.5394275784492493}

# with dropout

In [9]:
model_mlp_drop_ci = NeuralNetwork(input=3*32*32, x=0.2, y=0.2, first_hidden=3200, second_hidden=3000)
history_drop_ci, best_drop_ci = train_model(model=model_mlp_drop_ci, save_path="best_model_drop_ci.pt", train_dataset=Trainig_data_ci, epoches=25, lr=3e-4)
evaluate_model(model=model_mlp_drop_ci, load_path="weights_saved/best_model_drop_ci.pt", test_dataset=test_data_ci)

train:   0%|          | 0/625 [00:00<?, ?it/s]

val:   0%|          | 0/157 [00:00<?, ?it/s]

Training:   0%|          | 0/25 [00:00<?, ?it/s]

epoch 0 train | loss 1.8823 | Accuracy 0.3127 | Precision 0.3052 | F1 0.3062
epoch 0 val   | loss 1.8054 | Accuracy 0.3539 | Precision 0.3975 | F1 0.3374
new best val acc 0.3539, saved
epoch 1 train | loss 1.7224 | Accuracy 0.3829 | Precision 0.3764 | F1 0.3776
epoch 1 val   | loss 1.7002 | Accuracy 0.3946 | Precision 0.4101 | F1 0.3733
new best val acc 0.3946, saved
epoch 2 train | loss 1.6423 | Accuracy 0.4121 | Precision 0.4059 | F1 0.4068
epoch 2 val   | loss 1.6184 | Accuracy 0.4313 | Precision 0.4368 | F1 0.4168
new best val acc 0.4313, saved
epoch 3 train | loss 1.5874 | Accuracy 0.4326 | Precision 0.4259 | F1 0.4268
epoch 3 val   | loss 1.5688 | Accuracy 0.4462 | Precision 0.4599 | F1 0.4445
new best val acc 0.4462, saved
epoch 4 train | loss 1.5448 | Accuracy 0.4475 | Precision 0.4415 | F1 0.4423
epoch 4 val   | loss 1.5622 | Accuracy 0.4388 | Precision 0.4557 | F1 0.4239
epoch 5 train | loss 1.5181 | Accuracy 0.4549 | Precision 0.4490 | F1 0.4496
epoch 5 val   | loss 1.5153 |

{'loss': 1.3260339301103239,
 'acc': 0.5342999696731567,
 'prec': 0.5368273854255676,
 'f1': 0.5315027236938477}

# CNN class implementation

In [10]:
class ConvoFeatures(nn.Module):
    def __init__(self, in_channel=3, out_channel=32, second_out_channel=64, kernel_size=3,
                 stride=1, padding=0, dilation=1, groups=1, padding_mode="zeros",
                 first_drop=0, second_drop=0, layers=None):
        super().__init__()
        if layers is not None:                       # option 2: pass nn.Sequential
            self.conv_stack = layers
        else:
            self.conv_stack = nn.Sequential(
                nn.Conv2d(in_channels=in_channel, out_channels=out_channel, kernel_size=kernel_size,
                          stride=stride, padding=padding, dilation=dilation, groups=groups,
                          padding_mode=padding_mode),
                nn.ReLU(),
                nn.Dropout2d(first_drop),
                nn.MaxPool2d(2),
                nn.Conv2d(in_channels=out_channel, out_channels=second_out_channel,
                          kernel_size=kernel_size, stride=stride, padding=padding,
                          dilation=dilation, groups=groups, padding_mode=padding_mode),
                nn.ReLU(),
                nn.Dropout2d(second_drop),
                nn.MaxPool2d(2),
            )

    def forward(self, x):
        feature_map = self.conv_stack(x)
        return feature_map


class CNN(nn.Module):
    def __init__(self, image_size=32, in_channel=3, conv_kwargs=None, conv_layers=None, **mlp_kwargs):
        super().__init__()
        self.image_size = image_size
        self.in_channel = in_channel
        self.features = ConvoFeatures(in_channel=in_channel, layers=conv_layers, **(conv_kwargs or {}))
        n_features = self.count_features(self.features.conv_stack)
        self.classifier = NeuralNetwork(input=n_features, **mlp_kwargs)

    def forward(self, x):
        return self.classifier(self.features(x))

    def count_features(self, stack):
        """Run a dummy image through the conv stack and return the flattened size."""
        device = next(stack.parameters()).device
        dummy = torch.zeros(2, self.in_channel, self.image_size, self.image_size, device=device)
        try:
            with torch.no_grad():
                return stack(dummy).flatten(1).shape[1]
        except RuntimeError as e:
            raise ValueError(f"The conv layers don't fit a {self.image_size}x{self.image_size} "
                             f"image with {self.in_channel} channels: {e}") from None

    def _set_conv_stack(self, new_stack):
        n_features = self.count_features(new_stack)  # fails here if the shapes are impossible
        self.features.conv_stack = new_stack
        first = self.classifier.linear_stack[0]
        if first.in_features != n_features:          # flattened size changed -> new first Linear
            self.classifier.linear_stack[0] = nn.Linear(n_features, first.out_features).to(first.weight.device)
            self.classifier.input = n_features

    def add_conv_layer(self, position, layer, replace=False):
        self._set_conv_stack(edit_stack(self.features.conv_stack, position, layer, replace))

    def remove_conv_layer(self, position):
        self._set_conv_stack(edit_stack(self.features.conv_stack, position))

# Run 1 without dropout (FashionMNIST) (CNN)

In [11]:
model_fashion_cnn = CNN(image_size=28, in_channel=1)
history_fashion_cnn, best_fashion_cnn = train_model(model=model_fashion_cnn, save_path="best_model_CNN_FAshion.pt", epoches=10)
evaluate_model(model=model_fashion_cnn, load_path="weights_saved/best_model_CNN_FAshion.pt")

train:   0%|          | 0/750 [00:00<?, ?it/s]

val:   0%|          | 0/188 [00:00<?, ?it/s]

Training:   0%|          | 0/10 [00:00<?, ?it/s]

epoch 0 train | loss 0.4736 | Accuracy 0.8251 | Precision 0.8225 | F1 0.8225
epoch 0 val   | loss 0.3371 | Accuracy 0.8779 | Precision 0.8773 | F1 0.8770
new best val acc 0.8779, saved
epoch 1 train | loss 0.3034 | Accuracy 0.8868 | Precision 0.8863 | F1 0.8865
epoch 1 val   | loss 0.3229 | Accuracy 0.8825 | Precision 0.8839 | F1 0.8820
new best val acc 0.8825, saved
epoch 2 train | loss 0.2653 | Accuracy 0.9006 | Precision 0.9001 | F1 0.9003
epoch 2 val   | loss 0.3021 | Accuracy 0.8911 | Precision 0.8924 | F1 0.8902
new best val acc 0.8911, saved
epoch 3 train | loss 0.2339 | Accuracy 0.9129 | Precision 0.9125 | F1 0.9126
epoch 3 val   | loss 0.2848 | Accuracy 0.8976 | Precision 0.8980 | F1 0.8972
new best val acc 0.8976, saved
epoch 4 train | loss 0.2098 | Accuracy 0.9214 | Precision 0.9211 | F1 0.9212
epoch 4 val   | loss 0.2977 | Accuracy 0.9001 | Precision 0.9016 | F1 0.8997
new best val acc 0.9001, saved
epoch 5 train | loss 0.1822 | Accuracy 0.9303 | Precision 0.9301 | F1 0.930

{'loss': 0.3611640563341463,
 'acc': 0.9016000032424927,
 'prec': 0.901483952999115,
 'f1': 0.9013893008232117}

# Run 2 with dropout 0.2 (FashionMNIST)

In [12]:
model_fashion_cnn_drop = CNN(image_size=28, in_channel=1, x=0.2, y=0.2)
history_fashion_cnn_drop, best_fashion_cnn_drop = train_model(model=model_fashion_cnn_drop, save_path="best_model_CNN_FAshion_drop.pt", epoches=15)
evaluate_model(model=model_fashion_cnn_drop, load_path="weights_saved/best_model_CNN_FAshion_drop.pt")

train:   0%|          | 0/750 [00:00<?, ?it/s]

val:   0%|          | 0/188 [00:00<?, ?it/s]

Training:   0%|          | 0/15 [00:00<?, ?it/s]

epoch 0 train | loss 0.5042 | Accuracy 0.8106 | Precision 0.8068 | F1 0.8075
epoch 0 val   | loss 0.3795 | Accuracy 0.8563 | Precision 0.8753 | F1 0.8597
new best val acc 0.8563, saved
epoch 1 train | loss 0.3318 | Accuracy 0.8769 | Precision 0.8762 | F1 0.8764
epoch 1 val   | loss 0.3338 | Accuracy 0.8779 | Precision 0.8773 | F1 0.8731
new best val acc 0.8779, saved
epoch 2 train | loss 0.2940 | Accuracy 0.8923 | Precision 0.8917 | F1 0.8919
epoch 2 val   | loss 0.3209 | Accuracy 0.8834 | Precision 0.8841 | F1 0.8824
new best val acc 0.8834, saved
epoch 3 train | loss 0.2654 | Accuracy 0.8998 | Precision 0.8992 | F1 0.8994
epoch 3 val   | loss 0.3069 | Accuracy 0.8935 | Precision 0.8935 | F1 0.8906
new best val acc 0.8935, saved
epoch 4 train | loss 0.2458 | Accuracy 0.9078 | Precision 0.9073 | F1 0.9074
epoch 4 val   | loss 0.2854 | Accuracy 0.8963 | Precision 0.8970 | F1 0.8961
new best val acc 0.8963, saved
epoch 5 train | loss 0.2299 | Accuracy 0.9134 | Precision 0.9131 | F1 0.913

{'loss': 0.3003248651126388,
 'acc': 0.9038000106811523,
 'prec': 0.9037569165229797,
 'f1': 0.9034408330917358}

# CIFAR-10 (CNN)


In [13]:
model_cifar_cnn = CNN(image_size=32, in_channel=3)
history_cifar_cnn, best_cifar_cnn = train_model(model=model_cifar_cnn, save_path="best_model_CNN_CIFAR.pt", epoches=15, train_dataset=Trainig_data_ci, lr=3e-4)
evaluate_model(model=model_cifar_cnn, load_path="weights_saved/best_model_CNN_CIFAR.pt", test_dataset=test_data_ci)

train:   0%|          | 0/625 [00:00<?, ?it/s]

val:   0%|          | 0/157 [00:00<?, ?it/s]

Training:   0%|          | 0/15 [00:00<?, ?it/s]

epoch 0 train | loss 1.6755 | Accuracy 0.3820 | Precision 0.3730 | F1 0.3695
epoch 0 val   | loss 1.4969 | Accuracy 0.4562 | Precision 0.4855 | F1 0.4385
new best val acc 0.4562, saved
epoch 1 train | loss 1.3394 | Accuracy 0.5155 | Precision 0.5079 | F1 0.5097
epoch 1 val   | loss 1.2986 | Accuracy 0.5366 | Precision 0.5570 | F1 0.5354
new best val acc 0.5366, saved
epoch 2 train | loss 1.1802 | Accuracy 0.5767 | Precision 0.5710 | F1 0.5730
epoch 2 val   | loss 1.1446 | Accuracy 0.5948 | Precision 0.6041 | F1 0.5920
new best val acc 0.5948, saved
epoch 3 train | loss 1.0620 | Accuracy 0.6223 | Precision 0.6178 | F1 0.6195
epoch 3 val   | loss 1.0679 | Accuracy 0.6216 | Precision 0.6244 | F1 0.6138
new best val acc 0.6216, saved
epoch 4 train | loss 0.9767 | Accuracy 0.6559 | Precision 0.6526 | F1 0.6539
epoch 4 val   | loss 1.0399 | Accuracy 0.6316 | Precision 0.6474 | F1 0.6265
new best val acc 0.6316, saved
epoch 5 train | loss 0.8961 | Accuracy 0.6842 | Precision 0.6814 | F1 0.682

{'loss': 0.9290284191726879,
 'acc': 0.6979999542236328,
 'prec': 0.7011546492576599,
 'f1': 0.6971661448478699}

# CIFAR-10 (CNN)- dropout


In [14]:
model_cifar_cnn_drop = CNN(image_size=32, in_channel=3, x=0.2, y=0.2)
history_cifar_cnn_drop, best_cifar_cnn_drop = train_model(model=model_cifar_cnn_drop, save_path="best_model_CNN_CIFAR_drop.pt", epoches=15, train_dataset=Trainig_data_ci, lr=3e-4)
evaluate_model(model=model_cifar_cnn_drop, load_path="weights_saved/best_model_CNN_CIFAR_drop.pt", test_dataset=test_data_ci)

train:   0%|          | 0/625 [00:00<?, ?it/s]

val:   0%|          | 0/157 [00:00<?, ?it/s]

Training:   0%|          | 0/15 [00:00<?, ?it/s]

epoch 0 train | loss 1.7034 | Accuracy 0.3670 | Precision 0.3582 | F1 0.3574
epoch 0 val   | loss 1.4115 | Accuracy 0.4943 | Precision 0.4910 | F1 0.4863
new best val acc 0.4943, saved
epoch 1 train | loss 1.3460 | Accuracy 0.5118 | Precision 0.5049 | F1 0.5070
epoch 1 val   | loss 1.2851 | Accuracy 0.5462 | Precision 0.5445 | F1 0.5307
new best val acc 0.5462, saved
epoch 2 train | loss 1.1965 | Accuracy 0.5716 | Precision 0.5660 | F1 0.5680
epoch 2 val   | loss 1.1638 | Accuracy 0.5819 | Precision 0.5876 | F1 0.5778
new best val acc 0.5819, saved
epoch 3 train | loss 1.0876 | Accuracy 0.6126 | Precision 0.6080 | F1 0.6098
epoch 3 val   | loss 1.1144 | Accuracy 0.6026 | Precision 0.6172 | F1 0.5896
new best val acc 0.6026, saved
epoch 4 train | loss 1.0065 | Accuracy 0.6425 | Precision 0.6391 | F1 0.6404
epoch 4 val   | loss 1.0065 | Accuracy 0.6426 | Precision 0.6446 | F1 0.6407
new best val acc 0.6426, saved
epoch 5 train | loss 0.9253 | Accuracy 0.6747 | Precision 0.6718 | F1 0.673

{'loss': 0.9267320055870494,
 'acc': 0.7074999809265137,
 'prec': 0.7100744247436523,
 'f1': 0.705741822719574}

# CIFAR-10 (CNN)- dropout- weight decay


In [15]:
model_cifar_cnn_drop_decay = CNN(image_size=32, in_channel=3, x=0.2, y=0.2)
history_cifar_cnn_drop_decay, best_cifar_cnn_drop_decay = train_model(model=model_cifar_cnn_drop_decay, save_path="best_model_CNN_CIFAR_drop_weightdec.pt", epoches=15, train_dataset=Trainig_data_ci, lr=3e-4, optimizer_cls=torch.optim.AdamW, optimizer_kwargs={"weight_decay": 0.05})
evaluate_model(model=model_cifar_cnn_drop_decay, load_path="weights_saved/best_model_CNN_CIFAR_drop_weightdec.pt", test_dataset=test_data_ci)

train:   0%|          | 0/625 [00:00<?, ?it/s]

val:   0%|          | 0/157 [00:00<?, ?it/s]

Training:   0%|          | 0/15 [00:00<?, ?it/s]

epoch 0 train | loss 1.7249 | Accuracy 0.3627 | Precision 0.3542 | F1 0.3524
epoch 0 val   | loss 1.4528 | Accuracy 0.4691 | Precision 0.4686 | F1 0.4485
new best val acc 0.4691, saved
epoch 1 train | loss 1.3737 | Accuracy 0.5007 | Precision 0.4919 | F1 0.4944
epoch 1 val   | loss 1.3433 | Accuracy 0.5194 | Precision 0.5268 | F1 0.5025
new best val acc 0.5194, saved
epoch 2 train | loss 1.2241 | Accuracy 0.5588 | Precision 0.5524 | F1 0.5546
epoch 2 val   | loss 1.1976 | Accuracy 0.5755 | Precision 0.5953 | F1 0.5781
new best val acc 0.5755, saved
epoch 3 train | loss 1.1217 | Accuracy 0.5995 | Precision 0.5940 | F1 0.5961
epoch 3 val   | loss 1.1237 | Accuracy 0.6018 | Precision 0.6144 | F1 0.5974
new best val acc 0.6018, saved
epoch 4 train | loss 1.0228 | Accuracy 0.6342 | Precision 0.6302 | F1 0.6318
epoch 4 val   | loss 1.0330 | Accuracy 0.6332 | Precision 0.6411 | F1 0.6316
new best val acc 0.6332, saved
epoch 5 train | loss 0.9534 | Accuracy 0.6620 | Precision 0.6587 | F1 0.660

{'loss': 0.8688657841864665,
 'acc': 0.7107000350952148,
 'prec': 0.7140814065933228,
 'f1': 0.710512638092041}

# CIFAR-10 (CNN)- dropout- weight decay (also for Convo)

In [16]:
model_cifar_cnn_drop2d = CNN(image_size=32, in_channel=3, x=0.2, y=0.2, conv_kwargs={"first_drop": 0.2, "second_drop": 0.2})
history_cifar_cnn_drop_decay_drop2d, best_cifar_cnn_drop_decay_drop2d = train_model(model=model_cifar_cnn_drop2d, save_path="best_model_CNN_CIFAR_drop_weightdec.pt", epoches=24, train_dataset=Trainig_data_ci, lr=3e-4, optimizer_cls=torch.optim.AdamW, optimizer_kwargs={"weight_decay": 0.05})
evaluate_model(model=model_cifar_cnn_drop2d, load_path="weights_saved/best_model_CNN_CIFAR_drop_weightdec.pt", test_dataset=test_data_ci)

train:   0%|          | 0/625 [00:00<?, ?it/s]

val:   0%|          | 0/157 [00:00<?, ?it/s]

Training:   0%|          | 0/24 [00:00<?, ?it/s]

epoch 0 train | loss 1.8464 | Accuracy 0.3184 | Precision 0.3099 | F1 0.3049
epoch 0 val   | loss 1.5835 | Accuracy 0.4346 | Precision 0.4346 | F1 0.4185
new best val acc 0.4346, saved
epoch 1 train | loss 1.5324 | Accuracy 0.4473 | Precision 0.4391 | F1 0.4406
epoch 1 val   | loss 1.3823 | Accuracy 0.5023 | Precision 0.5118 | F1 0.4965
new best val acc 0.5023, saved
epoch 2 train | loss 1.3900 | Accuracy 0.5016 | Precision 0.4948 | F1 0.4969
epoch 2 val   | loss 1.2496 | Accuracy 0.5507 | Precision 0.5509 | F1 0.5407
new best val acc 0.5507, saved
epoch 3 train | loss 1.2878 | Accuracy 0.5422 | Precision 0.5361 | F1 0.5381
epoch 3 val   | loss 1.1664 | Accuracy 0.5853 | Precision 0.5815 | F1 0.5777
new best val acc 0.5853, saved
epoch 4 train | loss 1.1983 | Accuracy 0.5761 | Precision 0.5711 | F1 0.5729
epoch 4 val   | loss 1.1233 | Accuracy 0.6015 | Precision 0.6006 | F1 0.5927
new best val acc 0.6015, saved
epoch 5 train | loss 1.1323 | Accuracy 0.5981 | Precision 0.5936 | F1 0.595

{'loss': 0.8654600355275877,
 'acc': 0.7067999839782715,
 'prec': 0.7092276811599731,
 'f1': 0.7073574066162109}

# CIFAR-10 (CNN)- dropout- weight decay + zero padding

In [17]:
model_cifar_cnn_pad = CNN(image_size=32, in_channel=3, x=0.2, y=0.2, conv_kwargs={"first_drop": 0.2, "second_drop": 0.2, "padding": 1})
history_cifar_cnn_drop_decay_drop2d, best_cifar_cnn_drop_decay_drop2d = train_model(model=model_cifar_cnn_pad, save_path="best_model_CNN_CIFAR_drop_weightdec.pt", epoches=24, train_dataset=Trainig_data_ci, lr=3e-4, optimizer_cls=torch.optim.AdamW, optimizer_kwargs={"weight_decay": 0.05})
evaluate_model(model=model_cifar_cnn_pad, load_path="weights_saved/best_model_CNN_CIFAR_drop_weightdec.pt", test_dataset=test_data_ci)

train:   0%|          | 0/625 [00:00<?, ?it/s]

val:   0%|          | 0/157 [00:00<?, ?it/s]

Training:   0%|          | 0/24 [00:00<?, ?it/s]

epoch 0 train | loss 1.7899 | Accuracy 0.3442 | Precision 0.3375 | F1 0.3357
epoch 0 val   | loss 1.4760 | Accuracy 0.4716 | Precision 0.4736 | F1 0.4593
new best val acc 0.4716, saved
epoch 1 train | loss 1.4586 | Accuracy 0.4734 | Precision 0.4664 | F1 0.4682
epoch 1 val   | loss 1.3038 | Accuracy 0.5332 | Precision 0.5286 | F1 0.5233
new best val acc 0.5332, saved
epoch 2 train | loss 1.3031 | Accuracy 0.5285 | Precision 0.5226 | F1 0.5247
epoch 2 val   | loss 1.1862 | Accuracy 0.5766 | Precision 0.5894 | F1 0.5755
new best val acc 0.5766, saved
epoch 3 train | loss 1.1946 | Accuracy 0.5741 | Precision 0.5694 | F1 0.5711
epoch 3 val   | loss 1.0866 | Accuracy 0.6145 | Precision 0.6137 | F1 0.6088
new best val acc 0.6145, saved
epoch 4 train | loss 1.1078 | Accuracy 0.6056 | Precision 0.6013 | F1 0.6029
epoch 4 val   | loss 1.0373 | Accuracy 0.6259 | Precision 0.6296 | F1 0.6182
new best val acc 0.6259, saved
epoch 5 train | loss 1.0320 | Accuracy 0.6330 | Precision 0.6291 | F1 0.630

{'loss': 0.8304590508816349,
 'acc': 0.7211000323295593,
 'prec': 0.7192413806915283,
 'f1': 0.7197874784469604}

In [18]:
cifar10_conv_config = {
    "out_channel": 32,
    "second_out_channel": 64,
    "kernel_size": 3,
    "stride": 1,
    "padding": 1,
    "first_drop": 0.2,
    "second_drop": 0.3,
}


model = CNN(image_size=32, in_channel=3, conv_kwargs=cifar10_conv_config)

history_cifar_cnn_drop_decay_drop2d, best_cifar_cnn_drop_decay_drop2d = train_model(model=model, save_path="best_model_CNN_CIFAR_drop_weightdec_opt.pt", epoches=24, train_dataset= Trainig_data_ci, lr=3e-4, optimizer_cls=torch.optim.AdamW, optimizer_kwargs={"weight_decay": 0.05})
evaluate_model(model=model, load_path="weights_saved/best_model_CNN_CIFAR_drop_weightdec_opt.pt", test_dataset = test_data_ci)

train:   0%|          | 0/625 [00:00<?, ?it/s]

val:   0%|          | 0/157 [00:00<?, ?it/s]

Training:   0%|          | 0/24 [00:00<?, ?it/s]

epoch 0 train | loss 1.8032 | Accuracy 0.3405 | Precision 0.3336 | F1 0.3297
epoch 0 val   | loss 1.4712 | Accuracy 0.4698 | Precision 0.4712 | F1 0.4583
new best val acc 0.4698, saved
epoch 1 train | loss 1.4365 | Accuracy 0.4820 | Precision 0.4747 | F1 0.4763
epoch 1 val   | loss 1.2931 | Accuracy 0.5362 | Precision 0.5427 | F1 0.5338
new best val acc 0.5362, saved
epoch 2 train | loss 1.2899 | Accuracy 0.5399 | Precision 0.5341 | F1 0.5359
epoch 2 val   | loss 1.1782 | Accuracy 0.5780 | Precision 0.5838 | F1 0.5740
new best val acc 0.5780, saved
epoch 3 train | loss 1.1802 | Accuracy 0.5781 | Precision 0.5733 | F1 0.5750
epoch 3 val   | loss 1.0911 | Accuracy 0.6211 | Precision 0.6308 | F1 0.6224
new best val acc 0.6211, saved
epoch 4 train | loss 1.0910 | Accuracy 0.6126 | Precision 0.6090 | F1 0.6104
epoch 4 val   | loss 1.0214 | Accuracy 0.6342 | Precision 0.6419 | F1 0.6337
new best val acc 0.6342, saved
epoch 5 train | loss 1.0238 | Accuracy 0.6377 | Precision 0.6343 | F1 0.635

{'loss': 0.8688288012127967,
 'acc': 0.7122999429702759,
 'prec': 0.7126914858818054,
 'f1': 0.7099469900131226}

# Architecture: different depth

## MLP (FashionMNIST)

`print(model)` shows the positions. `add_layer(position, layer)` inserts, `add_layer(position, layer, replace=True)` overwrites, `remove_layer(position)` removes.

In [ ]:
model_depth_mlp = NeuralNetwork()
print(model_depth_mlp)


# model_depth_mlp.add_layer(6, nn.Linear(512, 512))
# model_depth_mlp.add_layer(7, nn.ReLU())
# model_depth_mlp.add_layer(8, nn.Dropout(0.0))
# model_depth_mlp.add_layer(1, nn.GELU(), replace=True)
# model_depth_mlp.remove_layer(2)


print(model_depth_mlp)
history_depth_mlp, best_depth_mlp = train_model(model=model_depth_mlp, save_path="best_model_depth_mlp.pt")
evaluate_model(model=model_depth_mlp, load_path="weights_saved/best_model_depth_mlp.pt")

## MLP (FashionMNIST), passing `nn.Sequential`

In [20]:
model_depth_mlp_seq = NeuralNetwork(layers=nn.Sequential(
    nn.Linear(28*28, 512),
    nn.ReLU(),
    nn.Linear(512, 316),
    nn.ReLU(),
    nn.Linear(316, 512),
    nn.ReLU(),
    nn.Linear(512, 28*28),
    nn.ReLU(),
    nn.Linear(28*28, 10),
))
print(model_depth_mlp_seq)
history_depth_mlp_seq, best_depth_mlp_seq = train_model(model=model_depth_mlp_seq, save_path="best_model_depth_mlp_seq.pt", epoches=30)
evaluate_model(model=model_depth_mlp_seq, load_path="weights_saved/best_model_depth_mlp_seq.pt")

NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=316, bias=True)
    (3): ReLU()
    (4): Linear(in_features=316, out_features=512, bias=True)
    (5): ReLU()
    (6): Linear(in_features=512, out_features=784, bias=True)
    (7): ReLU()
    (8): Linear(in_features=784, out_features=10, bias=True)
  )
)


train:   0%|          | 0/750 [00:00<?, ?it/s]

val:   0%|          | 0/188 [00:00<?, ?it/s]

Training:   0%|          | 0/30 [00:00<?, ?it/s]

epoch 0 train | loss 0.5715 | Accuracy 0.7901 | Precision 0.7868 | F1 0.7857
epoch 0 val   | loss 0.4407 | Accuracy 0.8420 | Precision 0.8456 | F1 0.8407
new best val acc 0.8420, saved
epoch 1 train | loss 0.4215 | Accuracy 0.8488 | Precision 0.8476 | F1 0.8472
epoch 1 val   | loss 0.4142 | Accuracy 0.8530 | Precision 0.8572 | F1 0.8521
new best val acc 0.8530, saved
epoch 2 train | loss 0.3843 | Accuracy 0.8615 | Precision 0.8604 | F1 0.8602
epoch 2 val   | loss 0.3781 | Accuracy 0.8675 | Precision 0.8707 | F1 0.8677
new best val acc 0.8675, saved
epoch 3 train | loss 0.3541 | Accuracy 0.8734 | Precision 0.8727 | F1 0.8727
epoch 3 val   | loss 0.3929 | Accuracy 0.8638 | Precision 0.8651 | F1 0.8620
epoch 4 train | loss 0.3392 | Accuracy 0.8783 | Precision 0.8779 | F1 0.8778
epoch 4 val   | loss 0.3806 | Accuracy 0.8606 | Precision 0.8616 | F1 0.8591
epoch 5 train | loss 0.3192 | Accuracy 0.8847 | Precision 0.8843 | F1 0.8842
epoch 5 val   | loss 0.3815 | Accuracy 0.8672 | Precision 0.

{'loss': 0.4439657701381073,
 'acc': 0.8848999738693237,
 'prec': 0.8849526643753052,
 'f1': 0.8845766186714172}

## CNN (CIFAR-10)

`add_conv_layer` / `remove_conv_layer` change the conv part; the classifier's input size is updated automatically. Use `model.classifier.add_layer(...)` for the fully connected part.

In [ ]:
model_depth_cnn = CNN(image_size=32, in_channel=3, x=0.2, y=0.2, conv_kwargs=cifar10_conv_config)
print(model_depth_cnn)


# model_depth_cnn.add_conv_layer(8, nn.Conv2d(64, 128, kernel_size=3, padding=1))
# model_depth_cnn.add_conv_layer(9, nn.ReLU())
# model_depth_cnn.add_conv_layer(10, nn.Dropout2d(0.3))
# model_depth_cnn.add_conv_layer(11, nn.MaxPool2d(2))
# model_depth_cnn.remove_conv_layer(2)

#  add classifier layers here
# model_depth_cnn.classifier.add_layer(6, nn.Linear(512, 512))
# model_depth_cnn.classifier.add_layer(7, nn.ReLU())


print(model_depth_cnn)
history_depth_cnn, best_depth_cnn = train_model(model=model_depth_cnn, save_path="best_model_depth_cnn.pt", epoches=24, train_dataset=Trainig_data_ci, lr=3e-4, optimizer_cls=torch.optim.AdamW, optimizer_kwargs={"weight_decay": 0.05})
evaluate_model(model=model_depth_cnn, load_path="weights_saved/best_model_depth_cnn.pt", test_dataset=test_data_ci)

## CNN (CIFAR-10), passing your own conv `nn.Sequential`

In [25]:
model_depth_cnn_seq = CNN(image_size=32, in_channel=3, x=0.2, y=0.2, conv_layers=nn.Sequential(
    nn.Conv2d(3, 32, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.Conv2d(32, 64, kernel_size=3, padding=1, stride=2),
    nn.ReLU(),
    nn.Conv2d(64, 128, kernel_size=3, padding=1, stride=2),
    nn.ReLU(),
    nn.Conv2d(128, 256, kernel_size=3, padding=1, stride=1),
    nn.ReLU(),
    nn.AvgPool2d(kernel_size=8, stride=8),
))
print(model_depth_cnn_seq)
history_depth_cnn_seq, best_depth_cnn_seq = train_model(model=model_depth_cnn_seq, save_path="best_model_depth_cnn_seq.pt", epoches=40, train_dataset=Trainig_data_ci, lr=3e-4 ,optimizer_cls=torch.optim.AdamW, optimizer_kwargs={"weight_decay": 0.05})
evaluate_model(model=model_depth_cnn_seq, load_path="weights_saved/best_model_depth_cnn_seq.pt", test_dataset=test_data_ci)

CNN(
  (features): ConvoFeatures(
    (conv_stack): Sequential(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): ReLU()
      (2): Conv2d(32, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
      (3): ReLU()
      (4): Conv2d(64, 128, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
      (5): ReLU()
      (6): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (7): ReLU()
      (8): AvgPool2d(kernel_size=8, stride=8, padding=0)
    )
  )
  (classifier): NeuralNetwork(
    (flatten): Flatten(start_dim=1, end_dim=-1)
    (linear_stack): Sequential(
      (0): Linear(in_features=256, out_features=512, bias=True)
      (1): ReLU()
      (2): Dropout(p=0.2, inplace=False)
      (3): Linear(in_features=512, out_features=512, bias=True)
      (4): ReLU()
      (5): Dropout(p=0.2, inplace=False)
      (6): Linear(in_features=512, out_features=10, bias=True)
    )
  )
)


train:   0%|          | 0/625 [00:00<?, ?it/s]

val:   0%|          | 0/157 [00:00<?, ?it/s]

Training:   0%|          | 0/40 [00:00<?, ?it/s]

epoch 0 train | loss 1.9562 | Accuracy 0.2381 | Precision 0.2358 | F1 0.2307
epoch 0 val   | loss 1.7901 | Accuracy 0.3019 | Precision 0.2871 | F1 0.2500
new best val acc 0.3019, saved
epoch 1 train | loss 1.6931 | Accuracy 0.3439 | Precision 0.3334 | F1 0.3327
epoch 1 val   | loss 1.6339 | Accuracy 0.3869 | Precision 0.3828 | F1 0.3644
new best val acc 0.3869, saved
epoch 2 train | loss 1.5627 | Accuracy 0.4067 | Precision 0.3944 | F1 0.3952
epoch 2 val   | loss 1.4817 | Accuracy 0.4526 | Precision 0.4599 | F1 0.4494
new best val acc 0.4526, saved
epoch 3 train | loss 1.4499 | Accuracy 0.4543 | Precision 0.4458 | F1 0.4473
epoch 3 val   | loss 1.4127 | Accuracy 0.4870 | Precision 0.5060 | F1 0.4894
new best val acc 0.4870, saved
epoch 4 train | loss 1.3529 | Accuracy 0.4944 | Precision 0.4867 | F1 0.4885
epoch 4 val   | loss 1.3038 | Accuracy 0.5200 | Precision 0.5112 | F1 0.5009
new best val acc 0.5200, saved
epoch 5 train | loss 1.2800 | Accuracy 0.5248 | Precision 0.5168 | F1 0.519

{'loss': 0.7693465148947042,
 'acc': 0.74590003490448,
 'prec': 0.7525129914283752,
 'f1': 0.744886040687561}